# Kaggle MMed-Llama Training - Working Configuration

**Problem Solved**: Using `peft 0.7.1 + accelerate 0.25.0` (no `clear_device_cache` needed)

This notebook uses older but fully compatible package versions that work on Kaggle without import errors.

In [ ]:
# CELL 1: Install Compatible Packages + Compile bitsandbytes for CUDA 12.8
# Time: ~10-15 minutes (includes compilation)
# ⚠️ Kaggle has CUDA 12.8 which requires compiling bitsandbytes from source

import subprocess
import sys
import os

print("🔧 Installing compatible package versions...")
print()

# Uninstall conflicting packages
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'transformers', 'peft', 'accelerate', 'bitsandbytes', 'trl'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Install base packages FIRST (without bitsandbytes)
packages = [
    'transformers==4.36.2',
    'peft==0.7.1',           # Works without clear_device_cache
    'accelerate==0.25.0',    # Compatible with peft 0.7.1
    'trl==0.4.7',            # OLD version without diffusers dependency
    'datasets==2.16.1',
    'scipy==1.11.4',
    'sentencepiece==0.1.99',
    'protobuf==4.25.1',
    'openpyxl==3.1.2',
    'ijson==3.2.3'           # For streaming JSON (avoid loading full 6GB)
]

print("📦 Installing base packages...")
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

print("✅ Base packages installed!")
print()

# ========== INSTALL BITSANDBYTES (CUDA 12.5 BINARIES FOR CUDA 12.8) ==========
print("=" * 80)
print("🔧 INSTALLING BITSANDBYTES WITH CUDA 12.5 COMPATIBILITY")
print("=" * 80)
print()
print("💡 Kaggle has CUDA 12.8, but we'll use CUDA 12.5 binaries (backward compatible)")
print()

# Set environment variable to use CUDA 12.5 binaries
os.environ['BNB_CUDA_VERSION'] = '125'

# Install bitsandbytes (will use 12.5 binaries which work with 12.8)
print("📦 Installing bitsandbytes...")
install_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'bitsandbytes>=0.44.0'],
    capture_output=True,
    text=True
)

if install_result.returncode == 0:
    print("✅ bitsandbytes installed successfully!")
    print()
    print("ℹ️  Using CUDA 12.5 binaries (compatible with CUDA 12.8)")
else:
    print("❌ Installation failed:")
    print(install_result.stderr[-300:] if len(install_result.stderr) > 300 else install_result.stderr)

print()
print("=" * 80)
print()

print("✅ All packages installed!")
print()
print("📦 Versions:")
print("   transformers: 4.36.2")
print("   peft: 0.7.1")
print("   accelerate: 0.25.0")
print("   bitsandbytes: >=0.44.0 (CUDA 12.5 binaries, compatible with 12.8)")
print("   trl: 0.4.7 (NO diffusers!)")
print()
print("⚠️  CLICK 'RESTART KERNEL' (⟳) NOW")
print()


In [ ]:
# CELL 2: Verify Installation (Run AFTER kernel restart)

import peft, accelerate, transformers, torch

print("✅ Package Verification:")
print(f"   peft: {peft.__version__}")
print(f"   accelerate: {accelerate.__version__}")
print(f"   transformers: {transformers.__version__}")
print(f"   torch: {torch.__version__}")
print()
print("🎉 All packages loaded successfully!")
print("   You can now proceed to training")
print()

## Dataset Extraction

Skip if you already have `/kaggle/working/training_data_combined_ALL.json`

In [ ]:
# CELL 3: Extract Datasets (Skip if already extracted)
# Time: ~45-90 minutes

import json, os, zipfile, re, random
from tqdm import tqdm
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import pandas as pd

def clean_text(text):
    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = re.sub(r' +', ' ', text)
    return text.strip()

# Check if already extracted
if os.path.exists("/kaggle/working/training_data_combined_ALL.json"):
    print("✅ Training data already extracted!")
    print("   Skipping extraction...")
    print()
    print("To re-extract, delete the file first:")
    print("   !rm /kaggle/working/training_data_combined_ALL.json")
else:
    print("📥 Starting dataset extraction...")
    print()
    
    # ==============================================================================
    # DATASET 1: MMEDC - ARABIC ONLY
    # ==============================================================================
    print("=" * 80)
    print("DATASET 1/4: MMEDC ARABIC")
    print("=" * 80)
    print()
    print("📥 Downloading Arabic.zip from HuggingFace (1.28 GB)...")
    
    zip_path = hf_hub_download(
        repo_id="Henrychur/MMedC",
        filename="Arabic.zip",
        repo_type="dataset"
    )
    
    print(f"✅ Downloaded to: {zip_path}")
    print("📦 Extracting Arabic medical texts...")
    
    mmedc_examples = []
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        txt_files = [f for f in zip_ref.namelist() if f.endswith('.txt')]
        print(f"📄 Found {len(txt_files):,} Arabic text files")
        
        for filename in tqdm(txt_files, desc="Processing MMedC"):
            try:
                with zip_ref.open(filename) as f:
                    content = f.read().decode('utf-8', errors='ignore')
                
                content = clean_text(content)
                if len(content) < 100:
                    continue
                
                # Chunk into 1500 char pieces with overlap
                chunk_size = 1500
                if len(content) > chunk_size:
                    for i in range(0, len(content), chunk_size):
                        chunk = content[i:i+chunk_size+200]
                        if len(chunk) >= 100:
                            mmedc_examples.append({
                                "input": "تعلم المعلومات الطبية التالية:",
                                "output": chunk,
                                "source": "MMedC"
                            })
                else:
                    mmedc_examples.append({
                        "input": "تعلم المعلومات الطبية التالية:",
                        "output": content,
                        "source": "MMedC"
                    })
            except:
                continue
    
    print(f"✅ MMedC Arabic: {len(mmedc_examples):,} examples")
    print()
    
    # ==============================================================================
    # DATASET 2: SHIFAA MEDICAL CONSULTATIONS
    # ==============================================================================
    print("=" * 80)
    print("DATASET 2/4: SHIFAA MEDICAL CONSULTATIONS")
    print("=" * 80)
    print()
    
    dataset = load_dataset("Ahmed-Selem/Shifaa_Arabic_Medical_Consultations")
    shifaa_medical_examples = []
    
    for split_name in dataset.keys():
        print(f"Processing split: {split_name}")
        for item in tqdm(dataset[split_name], desc=f"{split_name}"):
            question = clean_text(str(item.get('Question', '')))
            answer = clean_text(str(item.get('Answer', '')))
            
            if len(question) > 10 and len(answer) > 10:
                shifaa_medical_examples.append({
                    "input": question,
                    "output": answer,
                    "source": "Shifaa_Medical"
                })
    
    print(f"✅ Shifaa Medical: {len(shifaa_medical_examples):,} examples")
    print()
    
    # ==============================================================================
    # DATASET 3: SHIFAA MENTAL HEALTH CONSULTATIONS
    # ==============================================================================
    print("=" * 80)
    print("DATASET 3/4: SHIFAA MENTAL HEALTH CONSULTATIONS")
    print("=" * 80)
    print()
    
    dataset = load_dataset("Ahmed-Selem/Shifaa_Arabic_Mental_Health_Consultations")
    shifaa_mental_examples = []
    
    for split_name in dataset.keys():
        print(f"Processing split: {split_name}")
        for item in tqdm(dataset[split_name], desc=f"{split_name}"):
            question = clean_text(str(item.get('Question', '')))
            answer = clean_text(str(item.get('Answer', '')))
            
            if len(question) > 10 and len(answer) > 10:
                shifaa_mental_examples.append({
                    "input": question,
                    "output": answer,
                    "source": "Shifaa_Mental"
                })
    
    print(f"✅ Shifaa Mental Health: {len(shifaa_mental_examples):,} examples")
    print()
    
    # ==============================================================================
    # DATASET 4: AHD - ARABIC HEALTHCARE DATASET
    # ==============================================================================
    print("=" * 80)
    print("DATASET 4/4: AHD (Arabic Healthcare Dataset)")
    print("=" * 80)
    print()
    
    ahd_examples = []
    ahd_path = "/kaggle/input/ahd-dataset/AHD.xlsx"
    
    if os.path.exists(ahd_path):
        print(f"📂 Found AHD file: {ahd_path}")
        df = pd.read_excel(ahd_path)
        print(f"📊 Total rows: {len(df):,}")
        
        # Try different possible column names
        question_cols = ['question', 'Question', 'query', 'Query', 'q', 'Q']
        answer_cols = ['answer', 'Answer', 'response', 'Response', 'a', 'A']
        
        q_col = None
        a_col = None
        
        for col in question_cols:
            if col in df.columns:
                q_col = col
                break
        
        for col in answer_cols:
            if col in df.columns:
                a_col = col
                break
        
        if q_col and a_col:
            print(f"✅ Using columns: '{q_col}' and '{a_col}'")
            
            for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing AHD"):
                question = clean_text(str(row[q_col]))
                answer = clean_text(str(row[a_col]))
                
                if len(question) > 10 and len(answer) > 10:
                    ahd_examples.append({
                        "input": question,
                        "output": answer,
                        "source": "AHD"
                    })
            
            print(f"✅ AHD: {len(ahd_examples):,} examples")
        else:
            print(f"⚠️  Could not find question/answer columns")
    else:
        print(f"⚠️  AHD file not found at: {ahd_path}")
    
    print()
    
    # ==============================================================================
    # COMBINE ALL DATASETS
    # ==============================================================================
    print("=" * 80)
    print("COMBINING ALL DATASETS")
    print("=" * 80)
    print()
    
    all_examples = mmedc_examples + shifaa_medical_examples + shifaa_mental_examples + ahd_examples
    
    random.seed(42)
    random.shuffle(all_examples)
    
    print(f"📊 Final Dataset Statistics:")
    print(f"   1. MMedC Arabic:       {len(mmedc_examples):>8,} examples ({len(mmedc_examples)/len(all_examples)*100:>5.1f}%)")
    print(f"   2. Shifaa Medical:     {len(shifaa_medical_examples):>8,} examples ({len(shifaa_medical_examples)/len(all_examples)*100:>5.1f}%)")
    print(f"   3. Shifaa Mental:      {len(shifaa_mental_examples):>8,} examples ({len(shifaa_mental_examples)/len(all_examples)*100:>5.1f}%)")
    print(f"   4. AHD:                {len(ahd_examples):>8,} examples ({len(ahd_examples)/len(all_examples)*100:>5.1f}%)")
    print(f"   {'─' * 50}")
    print(f"   TOTAL:                 {len(all_examples):>8,} examples")
    print()
    
    # Save to JSON
    output_file = "/kaggle/working/training_data_combined_ALL.json"
    print(f"💾 Saving to: {output_file}")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_examples, f, ensure_ascii=False, indent=2)
    
    file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
    print(f"✅ Saved successfully!")
    print(f"📦 File size: {file_size_mb:.1f} MB")
    print()
    print("🎉 DATASET EXTRACTION COMPLETE!")


## Training with QLoRA

This uses the working package versions (peft 0.7.1 + accelerate 0.25.0)

In [ ]:
# CELL 4: Train MMed-Llama-3-8B with QLoRA (CHUNKED TRAINING)
# Time: ~2.2 hours per chunk, 27 chunks total (~60 hours total)
# 🔥 Trains on 100K chunks for safer checkpointing

import torch, json, glob, os, gc, time
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

print("=" * 80)
print("TRAINING MMED-LLAMA-3-8B WITH QLORA (CHUNKED)")
print("=" * 80)
print()

# ========== CONFIGURATION ==========
CHUNK_SIZE = 100_000  # Train on 100K examples at a time (safer checkpointing!)
TOTAL_EXAMPLES = 2_647_435  # ⚠️ From Cell 3 extraction (NO need to load JSON!)

print(f"⚙️  Configuration:")
print(f"   Chunk size: {CHUNK_SIZE:,} examples")
print(f"   Total examples: {TOTAL_EXAMPLES:,} (from extraction)")
print()

num_chunks = (TOTAL_EXAMPLES + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"✅ Will train in {num_chunks} chunks")
print(f"⏰ Time estimate: ~2.2 hours per chunk, ~{num_chunks * 2.2:.1f} hours total")
print(f"📅 Expected completion: 2 weeks (30h GPU quota per week)")
print()

# Track start time
start_time = time.time()
print(f"🕐 Training started at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🕐 Session will auto-stop at 12h mark (~{time.strftime('%H:%M:%S', time.localtime(start_time + 12*3600))})")
print()

# ========== LOAD MODEL (ONCE) ==========
print("🔄 Loading model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model_path = "/kaggle/input/medllm/models--Henrychur--MMed-Llama-3-8B"

print(f"   Loading from: {model_path}")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ Model loaded!")
print()

# Prepare for QLoRA
model.config.use_cache = False
model.config.pretraining_tp = 1
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print("✅ LoRA adapters attached")
print(f"   Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
print()

# ========== TRAIN ON CHUNKS ==========
def format_prompt(example):
    return {
        "text": f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

أنت نموذج لغوي طبي متخصص باللغة العربية. مهمتك هي تقديم معلومات طبية دقيقة ومفيدة.<|eot_id|><|start_header_id|>user<|end_header_id|>

{example['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{example['output']}<|eot_id|>"""
    }

print("🚀 Starting chunked training...")
print()

for chunk_idx in range(num_chunks):
    print("=" * 80)
    print(f"📦 CHUNK {chunk_idx + 1}/{num_chunks}")
    print("=" * 80)
    print()
    
    chunk_start_time = time.time()
    print(f"🕐 Chunk started at: {time.strftime('%H:%M:%S')}")
    
    # ⚠️ CRITICAL: Stream JSON and load ONLY this chunk (not entire file!)
    start_idx = chunk_idx * CHUNK_SIZE
    end_idx = min(start_idx + CHUNK_SIZE, TOTAL_EXAMPLES)
    
    print(f"📚 Streaming examples {start_idx:,} to {end_idx:,}...")
    
    # Stream JSON file and extract ONLY the chunk we need
    import ijson
    chunk_data = []
    current_idx = 0
    
    with open('/kaggle/working/training_data_combined_ALL.json', 'rb') as f:
        # Stream parse the JSON array
        parser = ijson.items(f, 'item')
        
        for item in parser:
            if current_idx >= end_idx:
                break  # Stop reading once we have our chunk
                
            if current_idx >= start_idx:
                chunk_data.append(item)  # Only keep items in our range
            
            current_idx += 1
    
    print(f"✅ Streamed {len(chunk_data):,} examples for this chunk (avoided loading full 6GB!)")
    print()
    
    # Format chunk data
    print("📝 Formatting chunk...")
    formatted_chunk = []
    for example in chunk_data:
        formatted_chunk.append(format_prompt(example))
    
    # Free chunk_data now that we have formatted version
    del chunk_data
    gc.collect()
    
    # Create dataset from chunk
    chunk_dataset = Dataset.from_list(formatted_chunk)
    print(f"✅ Chunk dataset ready: {len(chunk_dataset):,} examples")
    print()
    
    # Training args for this chunk
    training_args = TrainingArguments(
        output_dir="/kaggle/working/mmed_llama_qlora",
        num_train_epochs=1,                  # 1 pass through this chunk
        per_device_train_batch_size=2,       # Back to 2 since model fits in memory
        gradient_accumulation_steps=16,      # Back to 16 (effective batch=32)
        learning_rate=2e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        optim="paged_adamw_32bit",
        weight_decay=0.001,
        max_grad_norm=0.3,
        fp16=False,
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=10,
        save_steps=500,
        save_total_limit=2,
        evaluation_strategy="no",
        report_to="none",
        load_best_model_at_end=False,
        dataloader_num_workers=0,            # Set to 0 to avoid CPU bottleneck
        dataloader_pin_memory=True,
    )
    
    # Create trainer for this chunk
    trainer = SFTTrainer(
        model=model,
        train_dataset=chunk_dataset,
        peft_config=lora_config,
        dataset_text_field="text",
        max_seq_length=512,                  # Keep at 512 for speed
        tokenizer=tokenizer,
        args=training_args,
        packing=False,
    )
    
    print(f"🔄 Training on chunk {chunk_idx + 1}/{num_chunks}...")
    print()
    
    # Train on this chunk
    trainer.train()
    
    # Free memory after chunk
    del chunk_dataset, formatted_chunk, trainer
    gc.collect()
    torch.cuda.empty_cache()
    
    print()
    
    chunk_end_time = time.time()
    chunk_duration = chunk_end_time - chunk_start_time
    total_elapsed = chunk_end_time - start_time
    
    print(f"✅ Chunk {chunk_idx + 1}/{num_chunks} complete!")
    print(f"⏱️  Chunk took: {chunk_duration/60:.1f} minutes")
    print(f"⏱️  Total elapsed: {total_elapsed/3600:.1f} hours")
    print(f"📊 Progress: {(chunk_idx + 1)/num_chunks*100:.1f}% complete")
    
    # Estimate remaining time
    avg_time_per_chunk = total_elapsed / (chunk_idx + 1)
    remaining_chunks = num_chunks - (chunk_idx + 1)
    estimated_remaining = avg_time_per_chunk * remaining_chunks
    print(f"🕐 Estimated time remaining: {estimated_remaining/3600:.1f} hours")
    print()
    
    # Save checkpoint after each chunk
    checkpoint_dir = f"/kaggle/working/mmed_llama_qlora_chunk_{chunk_idx + 1}"
    model.save_pretrained(checkpoint_dir)
    tokenizer.save_pretrained(checkpoint_dir)
    print(f"💾 Checkpoint saved: {checkpoint_dir}")
    print()

# ========== FINAL SAVE ==========
print("=" * 80)
print("🎉 ALL CHUNKS COMPLETE!")
print("=" * 80)
print()

final_dir = "/kaggle/working/mmed_llama_qlora_final"
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)

print(f"✅ Final model saved to: {final_dir}")
print()
print("📊 Training Summary:")
print(f"   Total examples trained: {TOTAL_EXAMPLES:,}")
print(f"   Chunks processed: {num_chunks}")
print(f"   Chunk size: {CHUNK_SIZE:,}")
print()

# Cleanup
del model
gc.collect()
torch.cuda.empty_cache()

print("🎉 Training complete!")


## Resume Training (If Interrupted)

Use this if Kaggle stops after 12 hours

In [ ]:
# CELL 5: Resume from checkpoint

import glob

checkpoints = glob.glob("/kaggle/working/mmed_llama_qlora/checkpoint-*")
if checkpoints:
    latest = max(checkpoints, key=lambda x: int(x.split("-")[-1]))
    print(f"📂 Resuming from: {latest}")
    trainer.train(resume_from_checkpoint=latest)
else:
    print("❌ No checkpoints found!")

In [ ]:
# CELL 6: Resume Training from Last Completed Chunk
# Use this if Kaggle stopped in the middle of training

import torch, json, glob, os, gc
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer
from datasets import Dataset

print("=" * 80)
print("RESUMING TRAINING FROM LAST CHECKPOINT")
print("=" * 80)
print()

# ========== FIND LAST COMPLETED CHUNK ==========
chunk_dirs = glob.glob("/kaggle/working/mmed_llama_qlora_chunk_*")
if not chunk_dirs:
    print("❌ No checkpoint chunks found!")
    print("   Start training from Cell 4 instead")
else:
    # Extract chunk numbers and find the highest
    chunk_numbers = []
    for d in chunk_dirs:
        try:
            num = int(d.split("_chunk_")[-1])
            chunk_numbers.append(num)
        except:
            continue
    
    if not chunk_numbers:
        print("❌ No valid chunk checkpoints found!")
    else:
        last_chunk = max(chunk_numbers)
        checkpoint_dir = f"/kaggle/working/mmed_llama_qlora_chunk_{last_chunk}"
        
        print(f"✅ Found last completed chunk: {last_chunk}")
        print(f"📂 Loading from: {checkpoint_dir}")
        print()
        
        # ========== CONFIGURATION ==========
        CHUNK_SIZE = 100_000  # ⚠️ MUST match Cell 4 chunk size!
        TOTAL_EXAMPLES = 2_647_435
        START_FROM_CHUNK = last_chunk  # Resume from NEXT chunk
        
        num_chunks = (TOTAL_EXAMPLES + CHUNK_SIZE - 1) // CHUNK_SIZE
        
        print(f"⚙️  Configuration:")
        print(f"   Last completed chunk: {last_chunk}/{num_chunks}")
        print(f"   Will resume from chunk: {START_FROM_CHUNK + 1}/{num_chunks}")
        print(f"   Remaining chunks: {num_chunks - START_FROM_CHUNK}")
        print()
        
        # ========== LOAD BASE MODEL ==========
        print("🔄 Loading base model...")
        
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        
        model_path = "/kaggle/input/medllm/models--Henrychur--MMed-Llama-3-8B/snapshots/"
        snapshot_dirs = glob.glob(f"{model_path}*")
        if snapshot_dirs:
            model_path = snapshot_dirs[0]
        else:
            model_path = "Henrychur/MMed-Llama-3-8B"
        
        base_model = AutoModelForCausalLM.from_pretrained(
            model_path,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
        )
        
        tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"
        
        print("✅ Base model loaded!")
        print()
        
        # ========== LOAD LORA ADAPTERS FROM CHECKPOINT ==========
        print(f"🔄 Loading LoRA adapters from chunk {last_chunk}...")
        
        model = PeftModel.from_pretrained(base_model, checkpoint_dir)
        
        print("✅ LoRA adapters loaded from checkpoint!")
        print()
        
        # Prepare for continued training
        model.config.use_cache = False
        model.config.pretraining_tp = 1
        
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"✅ Model ready to resume training")
        print(f"   Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
        print()
        
        # ========== RESUME TRAINING FROM NEXT CHUNK ==========
        def format_prompt(example):
            return {
                "text": f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

أنت نموذج لغوي طبي متخصص باللغة العربية. مهمتك هي تقديم معلومات طبية دقيقة ومفيدة.<|eot_id|><|start_header_id|>user<|end_header_id|>

{example['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{example['output']}<|eot_id|>"""
            }
        
        print("🚀 Resuming chunked training...")
        print()
        
        # Start from the NEXT chunk after last completed
        for chunk_idx in range(START_FROM_CHUNK, num_chunks):
            print("=" * 80)
            print(f"📦 CHUNK {chunk_idx + 1}/{num_chunks}")
            print("=" * 80)
            print()
            
            # ⚠️ CRITICAL: Stream JSON and load ONLY this chunk (not entire file!)
            start_idx = chunk_idx * CHUNK_SIZE
            end_idx = min(start_idx + CHUNK_SIZE, TOTAL_EXAMPLES)
            
            print(f"📚 Streaming examples {start_idx:,} to {end_idx:,}...")
            
            # Stream JSON file and extract ONLY the chunk we need
            import ijson
            chunk_data = []
            current_idx = 0
            
            with open('/kaggle/working/training_data_combined_ALL.json', 'rb') as f:
                parser = ijson.items(f, 'item')
                
                for item in parser:
                    if current_idx >= end_idx:
                        break
                    if current_idx >= start_idx:
                        chunk_data.append(item)
                    current_idx += 1
            
            print(f"✅ Streamed {len(chunk_data):,} examples for this chunk")
            print()
            
            # Format chunk data
            print("📝 Formatting chunk...")
            formatted_chunk = []
            for example in chunk_data:
                formatted_chunk.append(format_prompt(example))
            
            del chunk_data
            gc.collect()
            
            # Create dataset from chunk
            chunk_dataset = Dataset.from_list(formatted_chunk)
            print(f"✅ Chunk dataset ready: {len(chunk_dataset):,} examples")
            print()
            
            # Training args for this chunk
            lora_config = LoraConfig(
                r=32,
                lora_alpha=64,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                                "gate_proj", "up_proj", "down_proj"],
                lora_dropout=0.05,
                bias="none",
                task_type="CAUSAL_LM"
            )
            
            training_args = TrainingArguments(
                output_dir="/kaggle/working/mmed_llama_qlora",
                num_train_epochs=1,
                per_device_train_batch_size=2,
                gradient_accumulation_steps=16,
                learning_rate=2e-5,
                lr_scheduler_type="cosine",
                warmup_ratio=0.03,
                optim="paged_adamw_32bit",
                weight_decay=0.001,
                max_grad_norm=0.3,
                fp16=False,
                bf16=True,
                gradient_checkpointing=True,
                logging_steps=10,
                save_steps=500,
                save_total_limit=2,
                evaluation_strategy="no",
                report_to="none",
                load_best_model_at_end=False,
                dataloader_num_workers=0,
            )
            
            # Create trainer for this chunk
            trainer = SFTTrainer(
                model=model,
                train_dataset=chunk_dataset,
                peft_config=lora_config,
                dataset_text_field="text",
                max_seq_length=1024,
                tokenizer=tokenizer,
                args=training_args,
                packing=False,
            )
            
            print(f"🔄 Training on chunk {chunk_idx + 1}/{num_chunks}...")
            print()
            
            # Train on this chunk
            trainer.train()
            
            # Free memory after chunk
            del chunk_dataset, formatted_chunk, trainer
            gc.collect()
            torch.cuda.empty_cache()
            
            print()
            print(f"✅ Chunk {chunk_idx + 1}/{num_chunks} complete!")
            print()
            
            # Save checkpoint after each chunk
            checkpoint_dir = f"/kaggle/working/mmed_llama_qlora_chunk_{chunk_idx + 1}"
            model.save_pretrained(checkpoint_dir)
            tokenizer.save_pretrained(checkpoint_dir)
            print(f"💾 Checkpoint saved: {checkpoint_dir}")
            print()
        
        # ========== FINAL SAVE ==========
        print("=" * 80)
        print("🎉 ALL REMAINING CHUNKS COMPLETE!")
        print("=" * 80)
        print()
        
        final_dir = "/kaggle/working/mmed_llama_qlora_final"
        model.save_pretrained(final_dir)
        tokenizer.save_pretrained(final_dir)
        
        print(f"✅ Final model saved to: {final_dir}")
        print()
        
        # Cleanup
        del model
        gc.collect()
        torch.cuda.empty_cache()
        
        print("🎉 Training complete!")
